Track implied volatility skew and term structure for SPX options, flag anomalies

In [11]:
%matplotlib inline

# For Plotly in Jupyter
import plotly.io as pio
pio.renderers.default = "notebook"
plt.ion()

In [23]:
import yfinance as yf
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import time
from mpl_toolkits.mplot3d import Axes3D
import warnings
from scipy.interpolate import UnivariateSpline, interp1d


In [24]:
warnings.filterwarnings('ignore')

In [41]:
try:
    get_ipython()
    IN_JUPYTER = True
except:
    IN_JUPYTER = False

class VolatilitySurfaceAnalyzer:
    """
    Analyze options implied volatility surface for a given ticker
    - Track skew (put/call IV differential)
    - Monitor term structure (front vs back month)
    - Flag anomalies vs historical norms
    """
    
    def __init__(self, ticker):
        self.ticker = ticker
        self.stock = yf.Ticker(ticker)
        self.current_price = None
        self.options_data = {}
        self.iv_surface = None
        
    def fetch_options_data(self):
        """Fetch all available options chains"""
        print(f"Fetching options data for {self.ticker}...")
        
        try:
            # Get current stock price
            hist = self.stock.history(period='1d')
            self.current_price = hist['Close'].iloc[-1]
            print(f"Current {self.ticker} price: ${self.current_price:.2f}")
            
            # Get all expiration dates
            expirations = self.stock.options
            print(f"Found {len(expirations)} expiration dates")
            
            # Fetch options for each expiration
            for exp_date in expirations[:8]:  # Limit to first 8 expirations for speed
                try:
                    opt_chain = self.stock.option_chain(exp_date)
                    
                    # Store calls and puts
                    self.options_data[exp_date] = {
                        'calls': opt_chain.calls,
                        'puts': opt_chain.puts,
                        'dte': self._calculate_dte(exp_date)
                    }
                    
                except Exception as e:
                    print(f"  Error fetching {exp_date}: {e}")
                    continue
            return True
            
        except Exception as e:
            return False
    
    def _calculate_dte(self, exp_date_str):
        """Calculate days to expiration"""
        exp_date = datetime.strptime(exp_date_str, '%Y-%m-%d')
        today = datetime.now()
        return (exp_date - today).days
    
    def build_iv_surface(self):
        """Build IV surface dataframe: strikes x expirations"""
        
        surface_data = []
        
        for exp_date, data in self.options_data.items():
            dte = data['dte']
            calls = data['calls']
            puts = data['puts']
            
            # Process calls
            for _, row in calls.iterrows():
                if row['impliedVolatility'] > 0 and row['volume'] > 0:
                    moneyness = row['strike'] / self.current_price
                    surface_data.append({
                        'expiration': exp_date,
                        'dte': dte,
                        'strike': row['strike'],
                        'moneyness': moneyness,
                        'iv': row['impliedVolatility'],
                        'type': 'call',
                        'volume': row['volume'],
                        'openInterest': row['openInterest']
                    })
            
            # Process puts
            for _, row in puts.iterrows():
                if row['impliedVolatility'] > 0 and row['volume'] > 0:
                    moneyness = row['strike'] / self.current_price
                    surface_data.append({
                        'expiration': exp_date,
                        'dte': dte,
                        'strike': row['strike'],
                        'moneyness': moneyness,
                        'iv': row['impliedVolatility'],
                        'type': 'put',
                        'volume': row['volume'],
                        'openInterest': row['openInterest']
                    })
        
        self.iv_surface = pd.DataFrame(surface_data)
        
        return self.iv_surface

    def analyze_skew(self):
        """Analyze put/call skew for each expiration"""
        print("VOLATILITY SKEW ANALYSIS")
        
        skew_results = []
        
        for exp_date, data in self.options_data.items():
            dte = data['dte']
            
            # Filter surface data for this expiration
            exp_data = self.iv_surface[self.iv_surface['expiration'] == exp_date]
            
            # Get ATM strikes (within 2% of spot)
            atm_mask = (exp_data['moneyness'] >= 0.98) & (exp_data['moneyness'] <= 1.02)
            
            # Get OTM puts (90-95% moneyness)
            otm_put_mask = (exp_data['moneyness'] >= 0.90) & (exp_data['moneyness'] <= 0.95) & (exp_data['type'] == 'put')
            
            # Get OTM calls (105-110% moneyness)
            otm_call_mask = (exp_data['moneyness'] >= 1.05) & (exp_data['moneyness'] <= 1.10) & (exp_data['type'] == 'call')
            
            if atm_mask.sum() > 0 and otm_put_mask.sum() > 0:
                atm_iv = exp_data[atm_mask]['iv'].mean()
                otm_put_iv = exp_data[otm_put_mask]['iv'].mean()
                otm_call_iv = exp_data[otm_call_mask]['iv'].mean() if otm_call_mask.sum() > 0 else np.nan
                
                # Calculate skew metrics
                put_skew = otm_put_iv - atm_iv  # Positive = puts expensive
                
                skew_results.append({
                    'expiration': exp_date,
                    'dte': dte,
                    'atm_iv': atm_iv,
                    'otm_put_iv': otm_put_iv,
                    'otm_call_iv': otm_call_iv,
                    'put_skew': put_skew,
                    'skew_ratio': otm_put_iv / atm_iv if atm_iv > 0 else np.nan
                })
        
        skew_df = pd.DataFrame(skew_results).sort_values('dte')
        
        # Display results
        print("\nSkew by Expiration:")
        print(skew_df[['expiration', 'dte', 'atm_iv', 'put_skew', 'skew_ratio']].to_string(index=False))
        
        # Flag anomalies
        print("SKEW ALERTS")
        
        avg_put_skew = skew_df['put_skew'].mean()
        std_put_skew = skew_df['put_skew'].std()
        
        for _, row in skew_df.iterrows():
            z_score = (row['put_skew'] - avg_put_skew) / std_put_skew if std_put_skew > 0 else 0
            
            if abs(z_score) > 1.5:
                direction = "STEEP" if z_score > 0 else "FLAT"
                print(f"\n  {row['expiration']} ({row['dte']} DTE): {direction} SKEW")
                print(f"   Put skew: {row['put_skew']:.4f} (Z-score: {z_score:.2f})")
                print(f"   ATM IV: {row['atm_iv']:.2%} | OTM Put IV: {row['otm_put_iv']:.2%}")
                
                if direction == "FLAT":
                    print(f"    TRADE IDEA: Skew unusually flat = complacency")
                    print(f"      → Buy OTM puts (cheap downside protection)")
                    print(f"      → Sell call spreads (collect premium)")
                else:
                    print(f"    TRADE IDEA: Skew unusually steep = fear premium")
                    print(f"      → Sell put spreads (rich put premium)")
                    print(f"      → Buy call spreads (calls relatively cheap)")
        
        return skew_df
    
    def analyze_term_structure(self):
        """Analyze term structure (front month vs back months)"""
        print("TERM STRUCTURE ANALYSIS")
        
        # Get ATM IV for each expiration
        term_structure = []
        
        for exp_date, data in self.options_data.items():
            dte = data['dte']
            exp_data = self.iv_surface[self.iv_surface['expiration'] == exp_date]
            
            # ATM options
            atm_mask = (exp_data['moneyness'] >= 0.98) & (exp_data['moneyness'] <= 1.02)
            
            if atm_mask.sum() > 0:
                atm_iv = exp_data[atm_mask]['iv'].mean()
                term_structure.append({
                    'expiration': exp_date,
                    'dte': dte,
                    'atm_iv': atm_iv
                })
        
        ts_df = pd.DataFrame(term_structure).sort_values('dte')
        
        print("\nATM IV by Expiration:")
        print(ts_df.to_string(index=False))
        
        # Check for inversion
        if len(ts_df) >= 2:
            front_iv = ts_df.iloc[0]['atm_iv']
            back_iv = ts_df.iloc[-1]['atm_iv']
            
            print(f"\nFront month IV: {front_iv:.2%}")
            print(f"Back month IV: {back_iv:.2%}")
            
            if front_iv > back_iv * 1.1:  # Front > 10% higher than back
                print("\n TERM STRUCTURE INVERTED!")
                print("   Market pricing near-term risk event")
                print("    TRADE IDEAS:")
                print("      → Calendar spread: Long front, short back (profit from inversion)")
                print("      → Directional: Hedge near-term risk with front-month puts")
                print("      → Wait for vol spike, then sell front month (mean reversion)")
            elif back_iv > front_iv * 1.2:  # Back > 20% higher
                print("\n STEEP TERM STRUCTURE")
                print("   Market pricing elevated future uncertainty")
                print("    TRADE IDEAS:")
                print("      → Sell back-month vol (rich premium)")
                print("      → Reverse calendar: Short front, long back")
        
        return ts_df
    
    def plot_volatility_surface_3d(self):
        """Plot interactive 3D volatility surface"""        
        # Check for Plotly
        try:
            import plotly.graph_objects as go
            use_plotly = True
        except ImportError:
            use_plotly = False
        
        # Prepare data for 3D plot
        pivot_data = self.iv_surface.pivot_table(
            values='iv',
            index='moneyness',
            columns='dte',
            aggfunc='mean'
        )
        
        if use_plotly:
            # Create interactive Plotly 3D surface
            fig = go.Figure(data=[go.Surface(
                x=pivot_data.columns,  # Days to expiration
                y=pivot_data.index,    # Moneyness
                z=pivot_data.values,   # Implied volatility
                colorscale='Viridis',
                colorbar=dict(title="Implied Vol", x=1.1),
                hovertemplate='<b>DTE</b>: %{x}<br>' +
                              '<b>Moneyness</b>: %{y:.2f}<br>' +
                              '<b>IV</b>: %{z:.2%}<br>' +
                              '<extra></extra>'
            )])
            
            fig.update_layout(
                title=dict(
                    text=f'{self.ticker} Interactive Volatility Surface<br>' +
                         f'<sub>Current Price: ${self.current_price:.2f} | Rotate with mouse, zoom with scroll</sub>',
                    x=0.5,
                    xanchor='center'
                ),
                scene=dict(
                    xaxis_title='Days to Expiration',
                    yaxis_title='Moneyness (Strike/Spot)',
                    zaxis_title='Implied Volatility',
                    camera=dict(
                        eye=dict(x=1.5, y=1.5, z=1.3)  # Initial viewing angle
                    )
                ),
                width=1200,
                height=800,
                font=dict(size=12)
            )
            
            # Display based on environment
            if IN_JUPYTER:
                fig.show()
            else:
                fig.show()
                
        else:
            # Fallback to matplotlib (non-interactive)
            X, Y = np.meshgrid(pivot_data.columns, pivot_data.index)
            Z = pivot_data.values
            
            fig = plt.figure(figsize=(14, 10))
            ax = fig.add_subplot(111, projection='3d')
            
            surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
            
            ax.set_xlabel('Days to Expiration', fontsize=12)
            ax.set_ylabel('Moneyness (Strike/Spot)', fontsize=12)
            ax.set_zlabel('Implied Volatility', fontsize=12)
            ax.set_title(f'{self.ticker} Volatility Surface\nCurrent Price: ${self.current_price:.2f}', 
                         fontsize=14, fontweight='bold')
            
            fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
            
            plt.tight_layout()
            
            if IN_JUPYTER:
                plt.show()
            else:
                plt.show(block=False)  # Non-blocking for scripts
                plt.pause(0.1)
    
    def plot_skew_by_expiration(self, skew_df):
        """Plot skew across different expirations (interactive)"""
        
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
            use_plotly = True
        except ImportError:
            use_plotly = False
        
        if use_plotly:
            # Create interactive Plotly subplots
            fig = make_subplots(
                rows=2, cols=1,
                subplot_titles=(
                    'Volatility Skew Term Structure',
                    'ATM vs OTM Put Implied Volatility'
                ),
                vertical_spacing=0.12
            )
            
            # Plot 1: Put skew over time
            fig.add_trace(
                go.Scatter(
                    x=skew_df['dte'],
                    y=skew_df['put_skew'],
                    mode='lines+markers',
                    name='Put Skew',
                    line=dict(color='darkred', width=3),
                    marker=dict(size=10),
                    hovertemplate='<b>DTE</b>: %{x}<br><b>Skew</b>: %{y:.4f}<extra></extra>'
                ),
                row=1, col=1
            )
            
            # Add mean line
            avg_skew = skew_df['put_skew'].mean()
            fig.add_hline(
                y=avg_skew,
                line_dash="dash",
                line_color="black",
                annotation_text="Average",
                row=1, col=1
            )
            
            # Add ±1 std dev band
            std_skew = skew_df['put_skew'].std()
            fig.add_hrect(
                y0=avg_skew - std_skew,
                y1=avg_skew + std_skew,
                fillcolor="gray",
                opacity=0.2,
                layer="below",
                line_width=0,
                row=1, col=1
            )
            
            # Plot 2: ATM vs OTM Put IV
            fig.add_trace(
                go.Scatter(
                    x=skew_df['dte'],
                    y=skew_df['atm_iv'],
                    mode='lines+markers',
                    name='ATM IV',
                    line=dict(width=3),
                    marker=dict(size=10),
                    hovertemplate='<b>DTE</b>: %{x}<br><b>ATM IV</b>: %{y:.2%}<extra></extra>'
                ),
                row=2, col=1
            )
            
            fig.add_trace(
                go.Scatter(
                    x=skew_df['dte'],
                    y=skew_df['otm_put_iv'],
                    mode='lines+markers',
                    name='OTM Put IV',
                    line=dict(width=3),
                    marker=dict(size=10, symbol='square'),
                    hovertemplate='<b>DTE</b>: %{x}<br><b>OTM Put IV</b>: %{y:.2%}<extra></extra>'
                ),
                row=2, col=1
            )
            
            # Update layout
            fig.update_xaxes(title_text="Days to Expiration", row=1, col=1)
            fig.update_xaxes(title_text="Days to Expiration", row=2, col=1)
            fig.update_yaxes(title_text="Put Skew (OTM Put IV - ATM IV)", row=1, col=1)
            fig.update_yaxes(title_text="Implied Volatility", tickformat=".1%", row=2, col=1)
            
            fig.update_layout(
                title_text=f'{self.ticker} Volatility Skew Analysis',
                height=900,
                showlegend=True,
                hovermode='x unified'
            )
            
            fig.show()
            
        else:
            # Fallback to matplotlib
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
            
            # Plot 1: Put skew over time
            ax1.plot(skew_df['dte'], skew_df['put_skew'], 'o-', linewidth=2, markersize=8, color='darkred')
            ax1.axhline(skew_df['put_skew'].mean(), color='black', linestyle='--', label='Average')
            ax1.fill_between(skew_df['dte'], 
                             skew_df['put_skew'].mean() - skew_df['put_skew'].std(),
                             skew_df['put_skew'].mean() + skew_df['put_skew'].std(),
                             alpha=0.2, color='gray', label='±1 Std Dev')
            ax1.set_xlabel('Days to Expiration', fontsize=12)
            ax1.set_ylabel('Put Skew (OTM Put IV - ATM IV)', fontsize=12)
            ax1.set_title(f'{self.ticker} Volatility Skew Term Structure', fontsize=14, fontweight='bold')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            # Plot 2: ATM vs OTM Put IV
            ax2.plot(skew_df['dte'], skew_df['atm_iv'], 'o-', label='ATM IV', linewidth=2, markersize=8)
            ax2.plot(skew_df['dte'], skew_df['otm_put_iv'], 's-', label='OTM Put IV', linewidth=2, markersize=8)
            ax2.set_xlabel('Days to Expiration', fontsize=12)
            ax2.set_ylabel('Implied Volatility', fontsize=12)
            ax2.set_title('ATM vs OTM Put Implied Volatility', fontsize=14, fontweight='bold')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
    
    def plot_term_structure(self, ts_df):
        """Plot ATM IV term structure (interactive)"""
        
        try:
            import plotly.graph_objects as go
            use_plotly = True
        except ImportError:
            use_plotly = False
        
        if use_plotly:
            # Create interactive Plotly chart
            fig = go.Figure()
            
            # Add main line
            fig.add_trace(go.Scatter(
                x=ts_df['dte'],
                y=ts_df['atm_iv'],
                mode='lines+markers',
                name='ATM IV',
                line=dict(color='darkblue', width=3),
                marker=dict(size=12),
                hovertemplate='<b>DTE</b>: %{x}<br><b>ATM IV</b>: %{y:.2%}<extra></extra>'
            ))
            
            # Check for inversion and highlight
            if len(ts_df) >= 2:
                front_iv = ts_df.iloc[0]['atm_iv']
                back_iv = ts_df.iloc[-1]['atm_iv']
                
                if front_iv > back_iv * 1.1:
                    # Highlight inverted region
                    fig.add_hrect(
                        y0=ts_df['atm_iv'].min(),
                        y1=ts_df['atm_iv'].max(),
                        fillcolor="red",
                        opacity=0.2,
                        layer="below",
                        line_width=0,
                        annotation_text="INVERTED (Front > Back)",
                        annotation_position="top left"
                    )
            
            fig.update_layout(
                title=f'{self.ticker} Volatility Term Structure',
                xaxis_title='Days to Expiration',
                yaxis_title='ATM Implied Volatility',
                yaxis_tickformat='.1%',
                height=600,
                width=1200,
                hovermode='x unified',
                font=dict(size=12)
            )
            
            fig.show()
            
        else:
            # Fallback to matplotlib
            fig, ax = plt.subplots(figsize=(14, 6))
            
            ax.plot(ts_df['dte'], ts_df['atm_iv'], 'o-', linewidth=2, markersize=10, color='darkblue')
            
            # Highlight inversion
            if len(ts_df) >= 2:
                if ts_df.iloc[0]['atm_iv'] > ts_df.iloc[-1]['atm_iv']:
                    ax.axhspan(ts_df['atm_iv'].min(), ts_df['atm_iv'].max(), alpha=0.2, color='red')
                    ax.text(ts_df['dte'].mean(), ts_df['atm_iv'].mean(), 
                           'INVERTED\n(Front > Back)', 
                           ha='center', va='center', fontsize=14, fontweight='bold', color='darkred')
            
            ax.set_xlabel('Days to Expiration', fontsize=12)
            ax.set_ylabel('ATM Implied Volatility', fontsize=12)
            ax.set_title(f'{self.ticker} Volatility Term Structure', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
    
    def generate_summary_report(self):
        """Generate executive summary of vol analysis"""
        print(f"{self.ticker} VOLATILITY SURFACE SUMMARY")
        print("="*70)
        
        # Overall stats
        print(f"\nCurrent Price: ${self.current_price:.2f}")
        print(f"Data Points: {len(self.iv_surface)}")
        print(f"Expirations Analyzed: {len(self.options_data)}")
        
        # Average IV by moneyness
        print("\nAverage IV by Moneyness:")
        print("  OTM Puts (90-95%): ", end="")
        otm_put = self.iv_surface[(self.iv_surface['moneyness'] >= 0.90) & 
                                   (self.iv_surface['moneyness'] <= 0.95) & 
                                   (self.iv_surface['type'] == 'put')]['iv'].mean()
        print(f"{otm_put:.2%}")
        
        print("  ATM (98-102%):     ", end="")
        atm = self.iv_surface[(self.iv_surface['moneyness'] >= 0.98) & 
                              (self.iv_surface['moneyness'] <= 1.02)]['iv'].mean()
        print(f"{atm:.2%}")
        
        print("  OTM Calls (105-110%): ", end="")
        otm_call = self.iv_surface[(self.iv_surface['moneyness'] >= 1.05) & 
                                    (self.iv_surface['moneyness'] <= 1.10) & 
                                    (self.iv_surface['type'] == 'call')]['iv'].mean()
        print(f"{otm_call:.2%}")
        
        print("\nOverall Skew: ", end="")
        overall_skew = otm_put - atm
        print(f"{overall_skew:.4f}", end="")
        if overall_skew > 0.05:
            print(" (STEEP - fear premium)")
        elif overall_skew < 0.02:
            print(" (FLAT - complacency)")
        else:
            print(" (NORMAL)")
        

In [43]:
# Initialize analyzer
analyzer = VolatilitySurfaceAnalyzer(ticker='TSLA')
    
    # Fetch options data
success = analyzer.fetch_options_data()
    
if success:
    # Build IV surface
    analyzer.build_iv_surface()

    # Analyze skew
    skew_df = analyzer.analyze_skew()
        
    # Analyze term structure
    ts_df = analyzer.analyze_term_structure()
        
    # Generate visualizations
    analyzer.plot_volatility_surface_3d()
    analyzer.plot_skew_by_expiration(skew_df)
    analyzer.plot_term_structure(ts_df)
        
    # Summary report
    analyzer.generate_summary_report()

Fetching options data for TSLA...
Current TSLA price: $438.07
Found 20 expiration dates
VOLATILITY SKEW ANALYSIS

Skew by Expiration:
expiration  dte   atm_iv  put_skew  skew_ratio
2026-01-09    4 0.448945  0.059306    1.132101
2026-01-16   11 0.425107  0.035158    1.082704
2026-01-23   18 0.409300  0.021150    1.051674
2026-01-30   25 0.498864  0.002575    1.005162
2026-02-06   32 0.493841 -0.000305    0.999382
2026-02-13   39 0.486852 -0.002124    0.995637
2026-02-20   46 0.476442 -0.005300    0.988876
2026-03-20   74 0.478488 -0.011375    0.976227
SKEW ALERTS

  2026-01-09 (4 DTE): STEEP SKEW
   Put skew: 0.0593 (Z-score: 1.93)
   ATM IV: 44.89% | OTM Put IV: 50.83%
    TRADE IDEA: Skew unusually steep = fear premium
      → Sell put spreads (rich put premium)
      → Buy call spreads (calls relatively cheap)
TERM STRUCTURE ANALYSIS

ATM IV by Expiration:
expiration  dte   atm_iv
2026-01-09    4 0.448945
2026-01-16   11 0.425107
2026-01-23   18 0.409300
2026-01-30   25 0.498864
2026

TSLA VOLATILITY SURFACE SUMMARY

Current Price: $438.07
Data Points: 1984
Expirations Analyzed: 8

Average IV by Moneyness:
  OTM Puts (90-95%): 47.83%
  ATM (98-102%):     46.04%
  OTM Calls (105-110%): 47.75%

Overall Skew: 0.0180 (FLAT - complacency)
